# Reading Large Files
### Do this some other time

In [4]:
LARGE_FILE = 'DELETE_ME_large_file.csv'

In [5]:
# create a large file: Took 10 minutes for 1GB

import csv
import random
import os

output_file = LARGE_FILE

# Target size in bytes (1 GB)
TARGET_SIZE = 1 * 1024 * 1024 * 1024

# Number of rows written per chunk
CHUNK_ROWS = 1_000_000

# Track current file size
def get_file_size(file_path):
    if os.path.exists(file_path):
        return os.path.getsize(file_path)
    return 0


with open(output_file, "w", newline="") as file:
    writer = csv.writer(file)

    # Write header
    writer.writerow(["age", "salary", "city_id"])

    while get_file_size(output_file) < TARGET_SIZE:

        rows = []
        for _ in range(CHUNK_ROWS):

            age = random.randint(20, 70)

            # Dummy columns
            salary = random.randint(20000, 150000)
            city_id = random.randint(1, 100)

            rows.append([age, salary, city_id])

        writer.writerows(rows)

        print(f"Current file size: {get_file_size(output_file) / (1024**3):.2f} GB")

print("CSV generation completed!")


Current file size: 0.01 GB
Current file size: 0.02 GB
Current file size: 0.04 GB
Current file size: 0.05 GB
Current file size: 0.06 GB
Current file size: 0.07 GB
Current file size: 0.09 GB
Current file size: 0.10 GB
Current file size: 0.11 GB
Current file size: 0.12 GB
Current file size: 0.14 GB
Current file size: 0.15 GB
Current file size: 0.16 GB
Current file size: 0.17 GB
Current file size: 0.19 GB
Current file size: 0.20 GB
Current file size: 0.21 GB
Current file size: 0.22 GB
Current file size: 0.24 GB
Current file size: 0.25 GB
Current file size: 0.26 GB
Current file size: 0.27 GB
Current file size: 0.28 GB
Current file size: 0.30 GB
Current file size: 0.31 GB
Current file size: 0.32 GB
Current file size: 0.33 GB
Current file size: 0.35 GB
Current file size: 0.36 GB
Current file size: 0.37 GB
Current file size: 0.38 GB
Current file size: 0.40 GB
Current file size: 0.41 GB
Current file size: 0.42 GB
Current file size: 0.43 GB
Current file size: 0.45 GB
Current file size: 0.46 GB
C

---

# Option 1 — Read CSV in Chunks (Most Common + Simple)

This is the **safest starting point**.

### 🔹 Why use it?

* Loads file in parts
* Prevents RAM overload
* Works well with pandas ML workflows

---

In [ ]:
import pandas as pd

chunk_size = 100000   # adjust based on RAM

chunks = []

for chunk in pd.read_csv(LARGE_FILE, chunksize=chunk_size):
    # Perform preprocessing here
    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)

---

### 🔹 Better Approach (Avoid storing full dataset)
Get only those records that are needed based on filters

Process chunk-wise:


In [ ]:
for chunk in pd.read_csv(LARGE_FILE, chunksize=100000):

    # Example: filter rows
    chunk = chunk[chunk["age"] > 30]

    # Example: save processed data
    chunk.to_csv("processed.csv", mode="a", index=False, header=False)


---

# Option 2 — Use Dask (Best for ML + Big Data)

Recommended if:

* You want pandas-like syntax
* Dataset is larger than RAM

---

### Install

```bash
pip install dask
```

---


In [ ]:
import dask.dataframe as dd

df = dd.read_csv(LARGE_FILE)

df = df[df["age"] > 30]

result = df.groupby("city_id").age.mean().compute()


---

### Advantages

* Handles large files easily
* Multi-core processing
* Almost same syntax as pandas


---

# Option 3 — Convert CSV → Parquet (Highly Recommended for ML)

CSV is slow because:

* Text based
* No compression
* No schema

Parquet is:

* Columnar
* Faster read
* Smaller storage
* Used in production ML pipelines

---


In [ ]:
import pandas as pd

df = pd.read_csv(LARGE_FILE)
df.to_parquet("data.parquet")

---

###  Later Read Fast

In [ ]:
df = pd.read_parquet("data.parquet")

# or read only the required columns

### This alone can improve performance **5-10x**

#  Option 4 — Use Polars ( Fastest Modern Option)

Polars is replacing pandas in many ML pipelines.

---

### Install

```bash
pip install polars
```

---


In [ ]:
import polars as pl

df = pl.read_csv(LARGE_FILE)

df_filtered = df.filter(pl.col("age") > 30)

### Polars is:

* Rust based
* Multi-threaded
* Extremely fast


---

# Option 5 — Load into Database (Best for Analytics)

If dataset is reused multiple times:

* Load into PostgreSQL / DuckDB / BigQuery
* Query using SQL

---


In [ ]:
### DuckDB Example (Very Powerful for CSV)

import duckdb

df = duckdb.query("""
    SELECT * FROM 'large_file.csv'
    WHERE age > 30
""").to_df()

### DuckDB is insanely fast for analytics.


In [ ]:
import os

os.remove(LARGE_FILE)